In [134]:
import pandas as pd
import requests
import time

In [135]:
from datetime import datetime
ANO_ATUAL = datetime.now().year

In [136]:
def resultados():
    lista_resultados = []
    rodada = 1
    while True:
        acesso = requests.get(f"https://api.jolpi.ca/ergast/f1/{ANO_ATUAL}/{rodada}/results/").json()
        races = acesso["MRData"]["RaceTable"]["Races"]

        if not races:
            break
        else:
            for race in races:
                results = race["Results"]
                circuito = race["Circuit"]["circuitId"]

                for result in results:
                    lista_resultados.append({
                        "temporada_atual": acesso["MRData"]["RaceTable"]["season"],
                        "rodada_atual": acesso["MRData"]["RaceTable"]["round"],
                        "id_circuito_atual": circuito,
                        "id_piloto_atual": result["Driver"]["driverId"],
                        "id_equipe_atual": result["Constructor"]["constructorId"],
                        "posicao_corrida_anterior": result["positionText"],
                        "posicao_ultima_corrida": result["position"],
                        "target": result["position"],
                        "grid_anterior": result["grid"],
                        "status": result["status"],
                        "pontos_anterior_individual": result["points"]
                    })
            rodada += 1
        time.sleep(1)
                    
    
    return lista_resultados


results = resultados()
df_inicio = pd.DataFrame(results)
df_inicio = df_inicio.astype({"temporada_atual": int, "rodada_atual": int}).sort_values(by=["temporada_atual", "rodada_atual"], ascending=True)

In [137]:
proxima_rodada = df_inicio[(df_inicio["temporada_atual"] == ANO_ATUAL)]["rodada_atual"].max()
count_abandono = df_inicio[(df_inicio["temporada_atual"] == ANO_ATUAL) & (df_inicio["rodada_atual"] <= proxima_rodada) & (df_inicio["posicao_corrida_anterior"] == "R") & (df_inicio["id_piloto_atual"] == "max_verstappen")]["posicao_corrida_anterior"]
pilotos = df_inicio[(df_inicio["temporada_atual"] == ANO_ATUAL) & (df_inicio["rodada_atual"] == proxima_rodada)]["id_piloto_atual"]
circuito = requests.get(f"https://api.jolpi.ca/ergast/f1/{ANO_ATUAL}/{proxima_rodada + 1}/circuits/").json()["MRData"]["CircuitTable"]["Circuits"][0]["circuitId"]
equipe = df_inicio[(df_inicio["temporada_atual"] == ANO_ATUAL) & (df_inicio["rodada_atual"] == proxima_rodada)]["id_equipe_atual"]

grid = []
infos_equipe = []
infos_piloto = []

acesso_results = requests.get(f"https://api.jolpi.ca/ergast/f1/{ANO_ATUAL}/{proxima_rodada}/results/").json()
results = acesso_results["MRData"]["RaceTable"]["Races"][0]["Results"]
for result in results:
    grid_1 = df_inicio[(df_inicio["rodada_atual"] <= proxima_rodada) & (df_inicio["temporada_atual"] == ANO_ATUAL) & (df_inicio["id_piloto_atual"] == result["Driver"]["driverId"])]["grid_anterior"]
    grid_1 = pd.to_numeric(grid_1, errors="coerce").tail(3).sum()

    posicao_1 = df_inicio[(df_inicio["rodada_atual"] <= proxima_rodada) & (df_inicio["temporada_atual"] == ANO_ATUAL) & (df_inicio["id_piloto_atual"] == result["Driver"]["driverId"])]["posicao_ultima_corrida"]
    posicao_1 = pd.to_numeric(posicao_1, errors="coerce").tail(3).sum()
    
    grid.append({
        "temporada_atual": int(acesso_results["MRData"]["RaceTable"]["season"]),
        "rodada_atual": int(acesso_results["MRData"]["RaceTable"]["round"]),
        "id_piloto_atual": result["Driver"]["driverId"],
        "grid_anterior": result["grid"],
        "posicao_ultima_corrida": int(result["position"]),
        "count_abandono": result["positionText"],
        "pontos_anterior_individual": result["points"],
        "media_posicao_ganha_anterior": f"{(grid_1 - posicao_1)/3:.2f}",
        "status": result["status"],
    })
df_grid = pd.DataFrame(grid)
df_grid["media_posicao_ganha_anterior"] = df_grid["media_posicao_ganha_anterior"].astype(float)

acesso_equipe = requests.get(f"https://api.jolpi.ca/ergast/f1/{ANO_ATUAL}/{proxima_rodada}/constructorstandings/").json()
teams = acesso_equipe["MRData"]["StandingsTable"]["StandingsLists"][0]["ConstructorStandings"]
for team in teams:
    infos_equipe.append({
        "temporada_atual": int(acesso_equipe["MRData"]["StandingsTable"]["season"]),
        "rodada_atual": int(acesso_equipe["MRData"]["StandingsTable"]["round"]),
        "id_equipe_atual": team["Constructor"]["constructorId"],
        "posicao_equipe_anterior": team["position"],
        "pontos_equipe_anterior": team["points"],
        "vitorias_equipe_anterior": team["wins"]
    })
df_team = pd.DataFrame(infos_equipe)

acesso_piloto = requests.get(f"https://api.jolpi.ca/ergast/f1/{ANO_ATUAL}/driverstandings/").json()
drivers = acesso_piloto["MRData"]["StandingsTable"]["StandingsLists"][0]["DriverStandings"]
for driver in drivers:
    df_posicao_1 = df_inicio[(df_inicio["rodada_atual"] <= proxima_rodada) & (df_inicio["temporada_atual"] == ANO_ATUAL) & (df_inicio["id_piloto_atual"] == driver["Driver"]["driverId"])]["posicao_ultima_corrida"]
    count_abandono = df_inicio[(df_inicio["temporada_atual"] == ANO_ATUAL) & (df_inicio["rodada_atual"] <= proxima_rodada) & (df_inicio["posicao_corrida_anterior"] == "R") & (df_inicio["id_piloto_atual"] == driver["Driver"]["driverId"])]["posicao_corrida_anterior"]

    infos_piloto.append({
        "temporada_atual": int(acesso_piloto["MRData"]["StandingsTable"]["season"]),
        "rodada_atual": proxima_rodada,
        "id_piloto_atual": driver["Driver"]["driverId"],
        "pontos_anterior": driver["points"],
        "posicao_camp_anterior": driver["position"],
        "num_vitorias_anterior": driver["wins"],
        "media_ultimas_3_anterior": f"{pd.to_numeric(df_posicao_1, errors="coerce").tail(3).reset_index(drop=True).mean():.2f}",
        "media_ultimas_5_anterior": f"{pd.to_numeric(df_posicao_1, errors="coerce").tail(5).reset_index(drop=True).mean():.2f}",
        "qtde_abandonos_anterior": pd.to_numeric(count_abandono, errors="coerce").count()
    })
df_driver = pd.DataFrame(infos_piloto)
df_driver["media_ultimas_3_anterior"] = df_driver["media_ultimas_3_anterior"].astype(float)
df_driver["media_ultimas_5_anterior"] = df_driver["media_ultimas_5_anterior"].astype(float)
df_driver["tendencia_desempenho"] = (df_driver["media_ultimas_3_anterior"] - df_driver["media_ultimas_5_anterior"]).round(2)

df = pd.DataFrame({
    'temporada_atual': ANO_ATUAL,
    'rodada_atual': proxima_rodada,
    'id_piloto_atual': pilotos,
    "id_circuito_atual": circuito,
    'id_equipe_atual': equipe,
})

In [138]:
df_merge1 = pd.merge(
    df,
    df_grid,
    on=["temporada_atual", "rodada_atual", "id_piloto_atual"],
    how="left"
)

df_merge1.head()

,temporada_atual,rodada_atual,id_piloto_atual,id_circuito_atual,id_equipe_atual,grid_anterior,posicao_ultima_corrida,count_abandono,pontos_anterior_individual,media_posicao_ganha_anterior,status
0,2026,6,antonelli,catalunya,mercedes,1,1,1,25,0.33,Finished
1,2026,6,hamilton,catalunya,ferrari,3,2,2,18,1.33,Finished
2,2026,6,gasly,catalunya,alpine,9,3,3,15,0.00,Finished
3,2026,6,hadjar,catalunya,red_bull,5,4,4,12,1.00,Finished
4,2026,6,piastri,catalunya,mclaren,7,5,5,10,-0.33,Finished


In [139]:
df_merge2 = pd.merge(
    df_merge1,
    df_team,
    on=["temporada_atual", "rodada_atual", "id_equipe_atual"],
    how="left"
)

df_merge2.head()

,temporada_atual,rodada_atual,id_piloto_atual,id_circuito_atual,id_equipe_atual,grid_anterior,posicao_ultima_corrida,count_abandono,pontos_anterior_individual,media_posicao_ganha_anterior,status,posicao_equipe_anterior,pontos_equipe_anterior,vitorias_equipe_anterior
0,2026,6,antonelli,catalunya,mercedes,1,1,1,25,0.33,Finished,1,244,6
1,2026,6,hamilton,catalunya,ferrari,3,2,2,18,1.33,Finished,2,165,0
2,2026,6,gasly,catalunya,alpine,9,3,3,15,0.00,Finished,5,50,0
3,2026,6,hadjar,catalunya,red_bull,5,4,4,12,1.00,Finished,4,69,0
4,2026,6,piastri,catalunya,mclaren,7,5,5,10,-0.33,Finished,3,116,0


In [140]:
df_merge3 = pd.merge(
    df_merge2,
    df_driver,
    on=["temporada_atual", "rodada_atual", "id_piloto_atual"],
    how="left"
)

df_merge3.head()

,temporada_atual,rodada_atual,id_piloto_atual,id_circuito_atual,id_equipe_atual,grid_anterior,posicao_ultima_corrida,count_abandono,pontos_anterior_individual,media_posicao_ganha_anterior,...,posicao_equipe_anterior,pontos_equipe_anterior,vitorias_equipe_anterior,pontos_anterior,posicao_camp_anterior,num_vitorias_anterior,media_ultimas_3_anterior,media_ultimas_5_anterior,qtde_abandonos_anterior,tendencia_desempenho
0,2026,6,antonelli,catalunya,mercedes,1,1,1,25,0.33,...,1,244,6,156,1,5,1.00,1.0,0,0.00
1,2026,6,hamilton,catalunya,ferrari,3,2,2,18,1.33,...,2,165,0,90,2,0,3.33,3.8,0,-0.47
2,2026,6,gasly,catalunya,alpine,9,3,3,15,0.00,...,5,50,0,35,8,0,10.67,9.0,0,1.67
3,2026,6,hadjar,catalunya,red_bull,5,4,4,12,1.00,...,4,69,0,26,9,0,10.33,10.2,0,0.13
4,2026,6,piastri,catalunya,mclaren,7,5,5,10,-0.33,...,3,116,0,58,5,0,6.33,8.0,0,-1.67


In [141]:
df_merge3.drop(columns=["count_abandono"], inplace=True)

In [142]:
import requests
from datetime import datetime, timezone

acesso_circuito = requests.get(f"https://api.jolpi.ca/ergast/f1/{ANO_ATUAL}/{proxima_rodada+1}/circuits/").json()
latitude = acesso_circuito["MRData"]["CircuitTable"]["Circuits"][0]["Location"]["lat"]
longetude = acesso_circuito["MRData"]["CircuitTable"]["Circuits"][0]["Location"]["long"]

data_corrida = requests.get(f"https://api.jolpi.ca/ergast/f1/{ANO_ATUAL}/{proxima_rodada+1}/").json()
hora = data_corrida["MRData"]["RaceTable"]["Races"][0]["time"]
data = data_corrida["MRData"]["RaceTable"]["Races"][0]["date"]

acesso_clima = requests.get(f"https://api.open-meteo.com/v1/forecast?latitude={latitude}&longitude={longetude}&start_date={data}&end_date={data}&hourly=temperature_2m,relative_humidity_2m,precipitation,shortwave_radiation,surface_temperature&timezone=auto").json()

dt = datetime.strptime(f"{data}T{hora}", "%Y-%m-%dT%H:%M:%SZ")
dt = dt.replace(tzinfo=timezone.utc)
openmeteo_time = dt.strftime("%Y-%m-%dT%H:%M")

indice_clima = 0
hourly = acesso_clima["hourly"]["time"]
for indice, valor in enumerate(hourly):
    if valor == openmeteo_time:
        indice_clima = indice
        break

temp_ar_media_api = acesso_clima["hourly"]["temperature_2m"][indice_clima:indice_clima+3]
temp_pista_media_api = acesso_clima["hourly"]["surface_temperature"][indice_clima:indice_clima+3]
umidade_media_api = acesso_clima["hourly"]["relative_humidity_2m"][indice_clima:indice_clima+3]
precipitation_api = acesso_clima["hourly"]["precipitation"][indice_clima:indice_clima+3]

temp_ar_media = f"{sum(temp_ar_media_api) / len(temp_ar_media_api):.2f}"
temp_pista_media = f"{sum(temp_pista_media_api) / len(temp_pista_media_api):.1f}"
umidade_media = f"{sum(umidade_media_api) / len(umidade_media_api):.2f}"
corrida_molhada = int(sum(precipitation_api) > 0)
perc_voltas_chuva = f"{len([x for x in precipitation_api if x > 0]) / len(precipitation_api) * 100:.2f}"

df_merge3["temp_ar_media"] = float(temp_ar_media)
df_merge3["temp_pista_media"] = float(temp_pista_media)
df_merge3["umidade_media"] = float(umidade_media)
df_merge3["corrida_molhada"] = corrida_molhada
df_merge3["perc_voltas_chuva"] = float(perc_voltas_chuva)

In [143]:
acesso_results = requests.get(f"https://api.jolpi.ca/ergast/f1/{ANO_ATUAL}/{proxima_rodada+1}/qualifying/").json()
quali_results = acesso_results["MRData"]["RaceTable"]["Races"]

def tempo_para_ms(tempo_str):
    if pd.isna(tempo_str):
        return None
    try:
        minutos, resto = tempo_str.split(':')
        segundos, ms = resto.split('.')
        return int(minutos) * 60000 + int(segundos) * 1000 + int(ms)
    except:
        return None

if not quali_results:
    pass
else:
    quali_temp = acesso_results["MRData"]["RaceTable"]["Races"][0]

    quali_results = acesso_results["MRData"]["RaceTable"]["Races"][0]["QualifyingResults"]
    info_quali = []
    
    for quali in quali_results:
        info_quali.append({
            "posicao_quali_atual": int(quali["position"]),
            "q1_atual": quali.get("Q1", None),
            "q2_atual": quali.get("Q2", None),
            "q3_atual": quali.get("Q3", None),
            "id_piloto_atual": quali["Driver"]["driverId"],
            "rodada_atual": proxima_rodada,
            "temporada_atual": int(quali_temp["season"])
        })
    df_quali = pd.DataFrame(info_quali)

    df_quali['q1_atual'] = df_quali['q1_atual'].apply(tempo_para_ms)
    df_quali['q2_atual'] = df_quali['q2_atual'].apply(tempo_para_ms)
    df_quali['q3_atual'] = df_quali['q3_atual'].apply(tempo_para_ms)

    menor_tempo = df_quali["q3_atual"].min()

    q3_rodada = []
    for value_q3 in df_quali['q3_atual']:
        if pd.notna(value_q3) and value_q3 != '':
            dif = value_q3 - menor_tempo
            q3_rodada.append(dif)
        else:
            q3_rodada.append(None)

    df_quali["dif_para_pole_atual"] = q3_rodada 

    df_merge3 = pd.merge(
        df_merge3,
        df_quali,
        on=["temporada_atual", "rodada_atual", "id_piloto_atual"],
        how="left"
    )
  

In [144]:
df_merge3["rodada_atual"] = proxima_rodada + 1

df_merge3.to_csv(r"DATA\analise.csv", index=False)